# 02. Pollutant Parameter Specification

This notebook reviews the Study 3 code and records what sources can support for PM2.5, CO2, nitrate-N, and lead in soil. The geographic scope is the four U.S. Census regions. This is a parameter review; it does not change the environment or set economic and reward formulas.

## Equations to consider

**PM2.5:** Represent annual exposure as background plus contributions from emissions: C[r,t] = C_background[r,t] + sum_s K[source, receptor, t] * E[s,t]. K must reflect atmospheric transport and removal. Do not carry annual concentration forward as pollutant mass. EPA's annual standard uses a three-year design value; WHO's annual guideline is a separate benchmark.

**CO2:** Keep one shared atmospheric perturbation. Each emissions pulse contributes its mass times the fraction remaining at its age: R(a) = 0.2173 + 0.2240 exp(-a/394.4) + 0.2824 exp(-a/36.54) + 0.2763 exp(-a/4.304). Age is in years. This comes from Joos et al. (2013). It needs consistent carbon units and a defined initial atmosphere.

**Nitrate-N:** Choose a catchment or water body and track mass and flow: M_next = M + inflow_load - outflow * concentration - transformation - treatment, with concentration = M / water_volume. The EPA drinking-water limit of 10 mg/L as N is not a general ecosystem threshold.

**Lead:** Track total soil lead and a mobile pool separately. Lead does not naturally decay. Immobilization shifts lead out of the mobile pool; excavation removes it from the modeled soil. Transport needs a defined pathway, such as erosion or runoff.

## Pollutant parameter table

The table records values only when a source supports a clear interpretation. The CO2 value is one coefficient in a published response function. The PM2.5, nitrate, and lead values are regulatory or health benchmarks, not selected simulation thresholds. Proposed scenario inputs remain labeled as assumptions.

In [1]:
import pandas as pd

pollutant_parameters = pd.DataFrame([
    {"pollutant": "PM2.5", "quantity / unit": "Population-weighted annual mean ambient concentration; µg/m³", "supported value": 9.0, "value meaning": "U.S. annual primary NAAQS; 3-year design value, not model failure threshold", "initial condition / annual emissions": "AQS annual monitor concentration; NEI source inventory emissions (tons/year). Proposed index 15 and input 12 are uncalibrated scenario units.", "equation": "C[r,t]=background+sum_s K[s→r,t]E[s,t]; K represents dispersion and deposition. Do not retain concentration as mass.", "prevention / cleanup": "Source control reduces emissions before release. Ambient cleanup needs technology-specific evidence.", "uncertainty / range": "AQS monitor and spatial coverage; derive uncertainty from selected monitor aggregation. WHO annual guideline 5 µg/m³ is a separate benchmark, not a range.", "sources": "https://aqs.epa.gov/aqsweb/airdata/download_files.html ; https://www.epa.gov/air-emissions-inventories/2020-national-emissions-inventory ; https://www.epa.gov/system/files/documents/2024-04/2024-pm-naaqs-fr-published.pdf ; https://www.who.int/publications/b/59593", "status": "Concentration benchmark sourced; region mapping and dynamics uncalibrated"},

    {"pollutant": "CO2", "quantity / unit": "Atmospheric excess CO2 mass (t CO2) or concentration anomaly (ppm); choose one", "supported value": 0.2173, "value meaning": "Permanent term in Joos multi-model mean IRF; other terms 0.2240/394.4 y, 0.2824/36.54 y, 0.2763/4.304 y", "initial condition / annual emissions": "EIA SEDS state energy-related emissions in million metric tons CO2/year; 2023 available. Initial anomaly/history undecided.", "equation": "ΔM(t)=sum_i E_i R(t−i); R(a)=0.2173+0.2240exp(−a/394.4)+0.2824exp(−a/36.54)+0.2763exp(−a/4.304). One shared global pool.", "prevention / cleanup": "Capture reduces source emissions; atmospheric removal is a distinct negative pulse requiring durable-storage/lifecycle specification.", "uncertainty / range": "Use paper's reported model spread for response uncertainty; code currently uses multi-model mean. No regional coefficient range selected.", "sources": "https://acp.copernicus.org/articles/13/2793/2013/ ; https://www.eia.gov/state/seds/sep_sum/html/pdf/sum_co2_tot.pdf", "status": "IRF literature-derived and implemented; emissions boundary and initial condition unresolved"},

    {"pollutant": "Nitrate-N (NO3-N)", "quantity / unit": "Dissolved concentration mg N/L plus water mass/load kg N and volume/flow m³", "supported value": 10.0, "value meaning": "EPA public drinking-water MCL, mg/L as N; not ambient ecosystem threshold", "initial condition / annual emissions": "EPA/USGS 2012 SPARROW reports state aggregated total nitrogen load (kg) and yield (kg/km²); TN is broader than nitrate-N and not a dynamic annual input.", "equation": "M_next=M+L_in−Q_out(M/V)−kM−treatment; concentration=M/V. Requires catchment, volume, flow, and transformation rates.", "prevention / cleanup": "Source-load prevention; water treatment is distinct from in-stream transformation and requires treatment compartment and residual pathway.", "uncertainty / range": "SPARROW has model and hydrologic uncertainty; annual loads vary around long-term estimates. Use model-specific uncertainty, not generic retention.", "sources": "https://www.epa.gov/nutrientpollution/estimated-total-nitrogen-and-total-phosphorus-loads-and-yields-generated-within ; https://water.usgs.gov/nawqa/sparrow/FAQs/faq.html ; https://nepis.epa.gov/Exe/ZyURL.cgi?Dockey=P10150PM.TXT", "status": "Load evidence available; nitrate fraction, water balance, transport and rates unresolved"},

    {"pollutant": "Lead (Pb) in soil", "quantity / unit": "Soil concentration mg Pb/kg dry soil and/or total/mobile mass per defined compartment", "supported value": 200.0, "value meaning": "EPA residential soil lead screening level in specified cleanup context; not universal threshold", "initial condition / annual emissions": "USGS/EPA background survey reports state geogenic surface-soil distributions (2007–2010); not total regional anthropogenic burden. Annual source load unselected.", "equation": "T_next=T+deposition+imports−physical removal−exports; no elemental decay. Mobile pool B changes via mobilization, immobilization, mobile removal, and pathway-specific export.", "prevention / cleanup": "Immobilization/containment changes mobility, not total Pb; excavation/removal reduces total mass.", "uncertainty / range": "Use state survey distributions for geogenic background, then separate urban/industrial contamination. Spatial variation is large; no universal regional range.", "sources": "https://www.epa.gov/superfund/usgs-background-soil-lead-survey-state-data ; https://www.epa.gov/superfund/residential-soil-lead-directive-cercla-sites-and-rcra-hazardous-waste-cleanup-facilities ; https://www.epa.gov/expobox/exposure-assessment-tools-chemical-classes-inorganics-and-fibers", "status": "Persistence and benchmark sourced; initial burden, mobile fraction and transport unresolved"},
])
display(pollutant_parameters)

,pollutant,quantity / unit,supported value,value meaning,initial condition / annual emissions,equation,prevention / cleanup,uncertainty / range,sources,status
0,PM2.5,Population-weighted annual mean ambient concen...,9.0000,U.S. annual primary NAAQS; 3-year design value...,AQS annual monitor concentration; NEI source i...,"C[r,t]=background+sum_s K[s→r,t]E[s,t]; K repr...",Source control reduces emissions before releas...,AQS monitor and spatial coverage; derive uncer...,https://aqs.epa.gov/aqsweb/airdata/download_fi...,Concentration benchmark sourced; region mappin...
1,CO2,Atmospheric excess CO2 mass (t CO2) or concent...,0.2173,Permanent term in Joos multi-model mean IRF; o...,EIA SEDS state energy-related emissions in mil...,ΔM(t)=sum_i E_i R(t−i); R(a)=0.2173+0.2240exp(...,Capture reduces source emissions; atmospheric ...,Use paper's reported model spread for response...,https://acp.copernicus.org/articles/13/2793/20...,IRF literature-derived and implemented; emissi...
2,Nitrate-N (NO3-N),Dissolved concentration mg N/L plus water mass...,10.0000,"EPA public drinking-water MCL, mg/L as N; not ...",EPA/USGS 2012 SPARROW reports state aggregated...,M_next=M+L_in−Q_out(M/V)−kM−treatment; concent...,Source-load prevention; water treatment is dis...,SPARROW has model and hydrologic uncertainty; ...,https://www.epa.gov/nutrientpollution/estimate...,"Load evidence available; nitrate fraction, wat..."
3,Lead (Pb) in soil,Soil concentration mg Pb/kg dry soil and/or to...,200.0000,EPA residential soil lead screening level in s...,USGS/EPA background survey reports state geoge...,T_next=T+deposition+imports−physical removal−e...,"Immobilization/containment changes mobility, n...",Use state survey distributions for geogenic ba...,https://www.epa.gov/superfund/usgs-background-...,Persistence and benchmark sourced; initial bur...


## Regional environmental inputs

The table uses Census regions and the provisional multipliers from the initial Study 3 proposal. They are dimensionless assumptions; no regional observations have been aggregated here. Census regions are not shared airsheds, watersheds, or soil compartments.

Emissions can be summed by mass across states, but concentrations need a monitor-weighting method. Nutrient transport follows watersheds, while CO2 contributes to one shared atmosphere. Economic starting values are not included.

In [2]:
regional_environment_parameters = pd.DataFrame([
    {"region": "Northeast", "states": "CT, ME, MA, NH, RI, VT, NJ, NY, PA", "PM2.5 multiplier": 0.9, "CO2 multiplier": 0.9, "nitrate multiplier": 0.8, "lead multiplier": 0.9, "environmental data basis": "No observations aggregated in this notebook; multipliers dimensionless.", "status": "Census geography; supplied multipliers provisional assumptions"},

    {"region": "Midwest", "states": "IL, IN, MI, OH, WI, IA, KS, MN, MO, NE, ND, SD", "PM2.5 multiplier": 1.2, "CO2 multiplier": 1.2, "nitrate multiplier": 1.4, "lead multiplier": 1.3, "environmental data basis": "No observations aggregated in this notebook; multipliers dimensionless.", "status": "Census geography; supplied multipliers provisional assumptions"},

    {"region": "South", "states": "DE, DC, FL, GA, MD, NC, SC, VA, WV, AL, KY, MS, TN, AR, LA, OK, TX", "PM2.5 multiplier": 1.1, "CO2 multiplier": 1.1, "nitrate multiplier": 1.2, "lead multiplier": 1.0, "environmental data basis": "No observations aggregated in this notebook; multipliers dimensionless.", "status": "Census geography; supplied multipliers provisional assumptions"},

    {"region": "West", "states": "AZ, CO, ID, MT, NV, NM, UT, WY, AK, CA, HI, OR, WA", "PM2.5 multiplier": 0.9, "CO2 multiplier": 1.0, "nitrate multiplier": 0.8, "lead multiplier": 0.8, "environmental data basis": "No observations aggregated in this notebook; multipliers dimensionless.", "status": "Census geography; supplied multipliers provisional assumptions"},
])
display(regional_environment_parameters)

,region,states,PM2.5 multiplier,CO2 multiplier,nitrate multiplier,lead multiplier,environmental data basis,status
0,Northeast,"CT, ME, MA, NH, RI, VT, NJ, NY, PA",0.9,0.9,0.8,0.9,No observations aggregated in this notebook; m...,Census geography; supplied multipliers provisi...
1,Midwest,"IL, IN, MI, OH, WI, IA, KS, MN, MO, NE, ND, SD",1.2,1.2,1.4,1.3,No observations aggregated in this notebook; m...,Census geography; supplied multipliers provisi...
2,South,"DE, DC, FL, GA, MD, NC, SC, VA, WV, AL, KY, MS...",1.1,1.1,1.2,1.0,No observations aggregated in this notebook; m...,Census geography; supplied multipliers provisi...
3,West,"AZ, CO, ID, MT, NV, NM, UT, WY, AK, CA, HI, OR...",0.9,1.0,0.8,0.8,No observations aggregated in this notebook; m...,Census geography; supplied multipliers provisi...


### What regional data can tell us

- **PM2.5:** EPA AQS has annual monitor records. EPA's emissions inventory has source emissions. Regional exposure estimates need a documented monitor and population-weighting method.
- **CO2:** EIA SEDS reports state energy-related emissions. These can be summed by Census region for a specific inventory year and boundary.
- **Nitrate-N:** EPA's SPARROW table reports total nitrogen loads and yields, not nitrate-N concentration. Catchment and flow data are needed for water dynamics.
- **Lead:** USGS/EPA soil surveys describe geogenic background lead. They do not represent all urban and industrial contamination or identify a universal mobile fraction.

Sources: [Census region definitions](https://www.census.gov/programs-surveys/economic-census/guidance-geographies/levels.html), [EPA AQS](https://aqs.epa.gov/aqsweb/airdata/download_files.html), [EPA emissions inventory](https://www.epa.gov/air-emissions-inventories/2020-national-emissions-inventory), [EIA state CO2 data](https://www.eia.gov/state/seds/sep_sum/html/pdf/sum_co2_tot.pdf), [SPARROW nitrogen estimates](https://www.epa.gov/nutrientpollution/estimated-total-nitrogen-and-total-phosphorus-loads-and-yields-generated-within), and [soil lead survey](https://www.epa.gov/superfund/usgs-background-soil-lead-survey-state-data).